# Neural Graphics Ex1: Training Your Own Diffusion Model!

## Setup environment

In [1]:
# We recommend using these utils.
# https://google.github.io/mediapy/mediapy.html
# https://einops.rocks/
!pip install mediapy einops wandb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.0 MB/s eta 0:00:00


In [2]:
# Import essential modules. Feel free to add whatever you need.
import torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from tqdm import tqdm

### Seed your work
To be able to reproduce your code, please use a random seed from this point onward.

In [3]:
def seed_everything(seed):
  torch.cuda.manual_seed(seed)
  torch.manual_seed(seed)

YOUR_SEED = 180 # modify if you want
seed_everything(YOUR_SEED)

### Experiment logging (Do Not Modify)

During **training** (Sections 2.3, 3.3, 4.1), metrics and eval images are logged to [Weights & Biases](https://wandb.ai/). Screenshot those for your PDF report.

Post-training experiments (3.4, 4.2) save figures locally — no W&B logging after `finish_run()`.

You do **not** need to implement any W&B code. Batch loss uses a training-step x-axis; epoch loss and in-training eval images use an epoch x-axis.

In [ ]:
# Do Not Modify — experiment logging utilities
import wandb
import torchvision

wandb.login()


def start_run(section: str, config: dict):
    run = wandb.init(project="neural-graphics-a2", name=section, config=config)
    wandb.define_metric("global_step", hidden=True)
    wandb.define_metric("epoch", hidden=True)
    wandb.define_metric("train/batch_loss", step_metric="global_step")
    wandb.define_metric("train/epoch_loss", step_metric="epoch")
    wandb.define_metric("eval/*", step_metric="epoch")
    return run


def log_batch_loss(loss: float, global_step: int):
    wandb.log({"global_step": global_step, "train/batch_loss": loss})


def log_epoch_loss(loss: float, epoch: int):
    wandb.log({"epoch": epoch, "train/epoch_loss": loss})


def log_sample_grid(samples: torch.Tensor, key: str, nrow: int = 5, **step_metrics):
    grid = torchvision.utils.make_grid(samples.cpu(), nrow=nrow, normalize=True)
    wandb.log({**step_metrics, key: wandb.Image(grid)})


def log_matplotlib_figure(fig, key: str, **step_metrics):
    wandb.log({**step_metrics, key: wandb.Image(fig)})


def plot_samples_with_labels(
    samples: torch.Tensor,
    labels: torch.Tensor,
    title: str,
    save_path: str | None = None,
    wandb_key: str | None = None,
    nrow: int = 5,
    **step_metrics,
):
    samples = samples.cpu()
    labels = labels.cpu()

    n = samples.shape[0]
    ncol = nrow
    nrows = (n + nrow - 1) // nrow

    fig, axes = plt.subplots(nrows, ncol, figsize=(ncol * 2, nrows * 2))
    axes = axes.flatten()

    for i in range(len(axes)):
        ax = axes[i]
        if i < n:
            ax.imshow(samples[i].squeeze(0), cmap="gray")
            ax.set_title(f"Label: {labels[i].item()}", fontsize=10)
        ax.axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path)
    if wandb_key is not None:
        log_matplotlib_figure(fig, wandb_key, **step_metrics)
    plt.close()


def finish_run():
    wandb.finish()

## 1. Basic Ops and UNet blocks
**Notations:**  
 * `Conv2D(kernel_size, stride, padding)` is `nn.Conv2d()`  
 * `BN` is `nn.BatchNorm2d()`  
 * `GELU` is `nn.GELU()`  
 * `ConvTranspose2D(kernel_size, stride, padding)` is `nn.ConvTranspose2d()`  
 * `AvgPool(kernel_size)` is `nn.AvgPool2d()`  
 * `Linear` is `nn.Linear()`  
 * `N`, `C`, `W` and `H` are batch size, channels num, weight and height respectively


### Basic Ops

In [4]:
class Conv(nn.Module):
    """
    A convolutional layer that doesn’t change the image
    resolution, only the channel dimension
    Applies nn.Conv2d(3, 1, 1) followed by BN and GELU.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes the Conv layer
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        #YOUR CODE HERE

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H, W) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()

class DownConv(nn.Module):
    """
        A convolutional layer downsamples the tensor by 2.
        The layer consists of Conv2D(3, 2, 1) followed by BN and GELU.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes the DownConv layer
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        #YOUR CODE HERE


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H/2, W/2) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()


class UpConv(nn.Module):
    """
    A convolutional layer that upsamples the tensor by 2.
    The layer consists of ConvTranspose2d(4, 2, 1) followed by
    BN and GELU.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes the UpConv layer
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        #YOUR CODE HERE


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H*2, W*2) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()


class Flatten(nn.Module):
    """
    Average pooling layer that flattens a 7x7 tensor into a 1x1 tensor.
    The layer consists of AvgPool followed by GELU.
    """
    def __init__(self):
        super().__init__()
        #YOUR CODE HERE

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, 7, 7) input tensor.

        Returns:
            (N, C, 1, 1) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()


class Unflatten(nn.Module):
    """
      Convolutional layer that unflattens/upsamples a 1x1 tensor into a
      7x7 tensor. The layer consists of ConvTranspose2D(7, 7, 0)
      followed by BN and GELU.
    """
    def __init__(self, in_channels: int):
        """
        Initializes Unflatten layer
        Args:
            in_channels (int): The number of input channels
        """
        super().__init__()
        #YOUR CODE HERE
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, 1, 1) input tensor.

        Returns:
            (N, in_channels, 7, 7) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()

class FC(nn.Module):
    """
    Fully connected layer, consisting of nn.linear followed by GELU.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes the FC layer
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        #YOUR CODE HERE
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels) input tensor.

        Returns:
            (N, out_channels) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()



### UNet Blocks

In [5]:
class ConvBlock(nn.Module):
    """
    Two consecutive Conv operations.
    Note that it has the same input and output shape as Conv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes ConvBlock
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        #YOUR CODE HERE

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H, W) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()


class DownBlock(nn.Module):
    """
    DownConv followed by ConvBlock. Note that it has the same input and output
    shape as DownConv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes DownBlock
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        #YOUR CODE HERE

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H/2, W/2) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()


class UpBlock(nn.Module):
    """
    UpConv followed by ConvBlock.
    Note that it has the same input and output shape as UpConv
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes UpBlock
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        #YOUR CODE HERE

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels, H, W) input tensor.

        Returns:
            (N, out_channels, H*2, W*2) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()

class FCBlock(nn.Module):
    """
    Fully-connected Block, consisting of FC layer followed by Linear layer. Note
    that it has the same input and output shape as FC.
    """
    def __init__(self, in_channels: int, out_channels: int):
        """
        Initializes FCBlock
        Args:
            in_channels (int): The number of input channels
            out_channels (int): The number of output channels
        """
        super().__init__()
        #YOUR CODE HERE

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_channels) input tensor.

        Returns:
            (N, out_channels) output tensor.
        """
        #YOUR CODE HERE
        raise NotImplementedError()

## 2. Unconditional Diffusion Framework


### 2.1 UNet architecture

In [ ]:
class DenoisingUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_hiddens: int
    ):
        super().__init__()
        #YOUR CODE HERE

    def forward(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            t: (N, 1) normalized time tensor.

        Returns:
            (N, C, H, W) output tensor.
        """
        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."
        #YOUR CODE HERE
        raise NotImplementedError()

### 2.1 DDPM Forward and Inverse Process


In [6]:
def ddpm_schedule(beta1: float, beta2: float, num_ts: int, device: str = 'cuda') -> dict:
    """Constants for DDPM training and sampling.

    Arguments:
        beta1: float, starting beta value.
        beta2: float, ending beta value.
        num_ts: int, number of timesteps.

    Returns:
        dict with keys:
            betas: linear schedule of betas from beta1 to beta2.
            alphas: 1 - betas.
            alpha_bars: cumulative product of alphas.
    """
    assert beta1 < beta2 < 1.0, "Expect beta1 < beta2 < 1.0."
    raise NotImplementedError()


In [ ]:
def ddpm_forward(
    unet: DenoisingUNet,
    ddpm_schedule: dict,
    x_0: torch.Tensor,
    num_ts: int,
) -> torch.Tensor:
    """Algorithm 1 of the DDPM paper (not including gradient step).

    Args:
        unet: DenoisingUNet
        ddpm_schedule: dict
        x_0: (N, C, H, W) input tensor.
        num_ts: int, number of timesteps.
    Returns:
        (,) diffusion loss.
    """
    unet.train()
    # YOUR CODE HERE.
    raise NotImplementedError()

In [ ]:
@torch.inference_mode()
def ddpm_sample(
    unet: DenoisingUNet,
    ddpm_schedule: dict,
    img_wh: tuple[int, int],
    batch_size: int,
    num_ts: int
) -> torch.Tensor:
    """Algorithm 2 of the DDPM paper.

    Args:
        unet: DenoisingUNet
        ddpm_schedule: dict
        img_wh: (H, W) output image width and height.
        num_ts: int, number of timesteps.

    Returns:
        (N, C, H, W) final sample.
    """
    unet.eval()
    # YOUR CODE HERE.

    raise NotImplementedError()

In [ ]:
# Do Not Modify
class DDPM(nn.Module):
    def __init__(
        self,
        unet: DenoisingUNet,
        betas: tuple[float, float] = (1e-4, 0.02),
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.betas_range = betas
        self.num_ts = num_ts
        self.p_uncond = p_uncond
        self.ddpm_schedule = ddpm_schedule(betas[0], betas[1], num_ts)

        for k, v in ddpm_schedule(betas[0], betas[1], num_ts).items():
            self.register_buffer(k, v, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.

        Returns:
            (,) diffusion loss.
        """
        return ddpm_forward(
            self.unet, self.ddpm_schedule, x, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        img_wh: tuple[int, int],
        batch_size: int
    ):
        return ddpm_sample(
            self.unet, self.ddpm_schedule, img_wh, batch_size, self.num_ts
        )

### 2.3 Train your denoiser

In [ ]:
# Hyper parameters - Modify if you wish
num_hidden = 128
batch_size = 64
num_epochs = 20
lr = 1e-3
img_wh = (28, 28)
eval_batch_size = 20
T = 300
eval_epochs = [1, 5, 10, 15, 20]

global_step = 0

# Init MNIST data loaders
train_data = MNIST(root='./data', train=True, download=True, transform=ToTensor())
test_data = MNIST(root='./data', train=False, download=True, transform=ToTensor())
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(test_data, batch_size=eval_batch_size, shuffle=True)

# Init denoiser and DDPM wrapper
denosier_unet = DenoisingUNet(in_channels=1, num_hiddens=num_hidden)
ddpm = DDPM(denosier_unet, num_ts=T)

# Optimizer and device setup - Adam optimizer with exponential learning rate decay
optimizer = torch.optim.Adam(ddpm.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(
    optimizer=optimizer, gamma=0.1 ** (1.0 / num_epochs)
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ddpm.to(device)

run = start_run(
    "ddpm",
    config=dict(
        num_hidden=num_hidden,
        batch_size=batch_size,
        num_epochs=num_epochs,
        lr=lr,
        T=T,
        eval_batch_size=eval_batch_size,
    ),
)

for epoch in tqdm(range(num_epochs)):
    ddpm.train()
    epoch_loss = 0.0

    for batch, (data, label) in enumerate(train_loader):
        optimizer.zero_grad()
        data = data.to(device)
        loss = ddpm(data)
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        epoch_loss += batch_loss

        log_batch_loss(batch_loss, global_step)
        global_step += 1

        if batch % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}], "
                f"Step [{batch}/{len(train_loader)}], "
                f"Loss: {batch_loss:.4f}"
            )

    avg_epoch_loss = epoch_loss / len(train_loader)
    log_epoch_loss(avg_epoch_loss, epoch + 1)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_epoch_loss:.4f}")
    scheduler.step()

    ddpm.eval()
    if epoch + 1 in eval_epochs:
        # YOUR EVAL CODE HERE.
        # Sample and log to W&B using
        # log_sample_grid(samples, f"eval/samples_epoch_{epoch+1}", epoch=epoch + 1)

        raise NotImplementedError()

finish_run()



## 3. Implementing class-conditioned diffusion framework with CFG


### 3.1 Adding Class-Conditioning to UNet architecture

In [26]:
class ConditionalDenoisingUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        num_hiddens: int,
    ):
        super().__init__()
        # YOUR CODE HERE.

    def forward(
        self,
        x: torch.Tensor,
        c: torch.Tensor,
        t: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N, num_classes) float condition tensor.
            t: (N, 1) normalized time tensor.
            mask: (N, 1) mask tensor. If not None, mask out condition when mask == 0.

        Returns:
            (N, C, H, W) output tensor.
        """
        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."
        # YOUR CODE HERE.
        raise NotImplementedError()

### 3.2 DDPM Forward and Inverse Process with CFG

In [27]:
def ddpm_forward(
    unet: ConditionalDenoisingUNet,
    ddpm_schedule: dict,
    x_0: torch.Tensor,
    c: torch.Tensor,
    p_uncond: float,
    num_ts: int,
) -> torch.Tensor:
    """Algorithm 3 (not including gradient step).

    Args:
        unet: ConditionalDenoisingUNet
        ddpm_schedule: dict
        x_0: (N, C, H, W) input tensor.
        c: (N,) int64 condition tensor.
        p_uncond: float, probability of unconditioning the condition.
        num_ts: int, number of timesteps.

    Returns:
        (,) diffusion loss.
    """
    unet.train()
    # YOUR CODE HERE.
    raise NotImplementedError()

In [24]:
@torch.inference_mode()
def ddpm_cfg_sample(
    unet: ConditionalDenoisingUNet,
    ddpm_schedule: dict,
    c: torch.Tensor,
    img_wh: tuple[int, int],
    num_ts: int,
    guidance_scale: float = 5.0
) -> torch.Tensor:
    """Algorithm 4.

    Args:
        unet: ConditionalDenoisingUNet
        ddpm_schedule: dict
        c: (N,) int64 condition tensor. Only for class-conditional
        img_wh: (H, W) output image width and height.
        num_ts: int, number of timesteps.
        guidance_scale: float, CFG scale.

    Returns:
        (N, C, H, W) final sample.
    """
    unet.eval()
    # YOUR CODE HERE.
    raise NotImplementedError()

In [14]:
# Do Not Modify
class DDPM(nn.Module):
    def __init__(
        self,
        unet: ConditionalDenoisingUNet,
        betas: tuple[float, float] = (1e-4, 0.02),
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.betas = betas
        self.num_ts = num_ts
        self.p_uncond = p_uncond
        self.ddpm_schedule = ddpm_schedule(betas[0], betas[1], num_ts)

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N,) int64 condition tensor.

        Returns:
            (,) diffusion loss.
        """
        return ddpm_forward(
            self.unet, self.ddpm_schedule, x, c, self.p_uncond, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        c: torch.Tensor,
        img_wh: tuple[int, int],
        guidance_scale: float = 5.0
    ):
        return ddpm_cfg_sample(
            self.unet, self.ddpm_schedule, c, img_wh, self.num_ts, guidance_scale
        )

### 3.3 Train your class-conditioned denoiser

In [ ]:
# YOUR CODE HERE.
# Train a class-conditioned DDPM (mirror Section 2.3):
#   - for W&B logging, use start_run("conditional-ddpm", ...) / log_batch_loss / finish_run
#   - eval at epochs [1,5,10,15,20]
#   - you may use plot_samples_with_labels to log samples to W&B

### 3.4 Experiment with different guidance sacles

In [ ]:
# YOUR CODE HERE.
# Experiment with guidance scales [0, 1, 5, 7, 10, 15]:
#   - Generate one sample per digit (0-9) using ddpm_cfg_sample(...)
#   - Save labeled grids with plot_samples_with_labels(..., save_path=f"guidance_scale_{gamma}.png")
#   - No W&B logging needed here

## 4. Flow Matching (Bonus)

This entire section is optional bonus material. If you choose to complete it, you will implement and train a Flow Matching model on MNIST.

Unlike DDPM, which learns to predict the noise added to an image, Flow Matching learns a velocity field that moves samples from a simple noise distribution toward the data distribution.

You will reuse the same `DenoisingUNet` architecture from Section 2. No architectural changes are required.

### 4.1 Flow Matching Training Objective

In [ ]:
def flow_matching_forward(
    unet: DenoisingUNet,
    x_1: torch.Tensor,
) -> torch.Tensor:
    """
    Flow Matching training objective.

    Args:
        unet: DenoisingUNet.
        x_1: (N, C, H, W) clean images from the dataset.

    Returns:
        (,) flow matching loss.
    """
    unet.train()

    # YOUR CODE HERE:

    raise NotImplementedError()

### 4.2 Flow Matching Sampling

In [ ]:
@torch.inference_mode()
def flow_matching_sample(
    unet: DenoisingUNet,
    img_wh: tuple[int, int],
    batch_size: int,
    num_steps: int,
) -> torch.Tensor:
    """
    Flow Matching sampling using Euler integration.

    Args:
        unet: DenoisingUNet.
        img_wh: (H, W) output image size.
        batch_size: number of samples to generate.
        num_steps: number of Euler integration steps.

    Returns:
        (N, C, H, W) generated samples.
    """
    unet.eval()

    # YOUR CODE HERE:

    raise NotImplementedError()

In [ ]:
# Do Not Modify
class FlowMatching(nn.Module):
    def __init__(
        self,
        unet: DenoisingUNet,
        num_steps: int = 50,
    ):
        super().__init__()
        self.unet = unet
        self.num_steps = num_steps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return flow_matching_forward(self.unet, x)

    @torch.inference_mode()
    def sample(
        self,
        img_wh: tuple[int, int],
        batch_size: int,
    ) -> torch.Tensor:
        return flow_matching_sample(
            self.unet,
            img_wh,
            batch_size,
            self.num_steps,
        )

### 4.3 Train your Flow Matching model

Train the Flow Matching model on MNIST. Reuse the same `DenoisingUNet` and hyperparameters as Section 2.3 (`num_steps=50` for Euler sampling is the main extra setting).

Submit screenshots from your W&B run:

1. Training loss curves (`train/batch_loss` and `train/epoch_loss`).
2. Evaluation sample grids at epochs 1, 5, 10, 15, and 20.

In [ ]:
# YOUR CODE HERE.
# Train a Flow Matching model (mirror Section 2.3):
#   - Use the same hyperparameters as Section 2.3; set num_steps=50 for Euler sampling
#   - eval at epochs [1, 5, 10, 15, 20]
#   - for W&B logging, use start_run("flow-matching", ...) / log_batch_loss / log_sample_grid / finish_run()